# XGBoost Text Normalization Experiments

Compares Bag-of-Words + XGBoost with two text normalization strategies:
basic lowercase normalization and English stop-word removal that preserves negation tokens.
Non-English rows keep basic normalization so all 15 competition languages stay supported.

In [ ]:
import json
from pathlib import Path

import joblib
import mlflow
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

from helpers.data import load_processed_data, load_sample_submission
from helpers.metrics import compute_all_metrics
from helpers.mlflow import log_common_context, log_metrics, log_model_artifacts, start_notebook_run
from helpers.submission import build_submission, save_submission

## Load Data

Prefer DVC-versioned processed splits. If this notebook is copied into Kaggle, fallback paths can be used.

In [ ]:
PROCESSED_DIR = Path('../data/processed')
RAW_DIR = Path('../data/raw')
KAGGLE_INPUT_DIR = Path('/kaggle/input/competitions/contradictory-my-dear-watson')

if (PROCESSED_DIR / 'train_split.csv').exists():
    train_df, val_df, test_df = load_processed_data(PROCESSED_DIR)
    sample_submission = load_sample_submission(RAW_DIR / 'sample_submission.csv')
    data_source = 'dvc_processed_split'
elif (KAGGLE_INPUT_DIR / 'train.csv').exists():
    full_train_df = pd.read_csv(KAGGLE_INPUT_DIR / 'train.csv')
    test_df = pd.read_csv(KAGGLE_INPUT_DIR / 'test.csv')
    sample_submission = pd.read_csv(KAGGLE_INPUT_DIR / 'sample_submission.csv')
    train_df, val_df = train_test_split(
        full_train_df,
        test_size=0.2,
        random_state=42,
        stratify=full_train_df['label'],
    )
    data_source = 'kaggle_input_fallback_split'
else:
    raise FileNotFoundError('Could not find DVC processed splits or Kaggle input files.')

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f'Data loaded. Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}, source: {data_source}')

## Text Normalization Strategies

In [ ]:
def normalize_basic(value):
    return ' '.join(str(value).lower().split())


NEGATION_WORDS = {'not', 'no', 'never', 'neither', 'nor', 'none', 'cannot'}
CUSTOM_STOP_WORDS = [word for word in ENGLISH_STOP_WORDS if word not in NEGATION_WORDS]


def normalize_with_negation(row, text_column):
    text = str(row[text_column]).lower()
    lang = row['lang_abv']

    if lang == 'en':
        words = text.split()
        filtered_words = [word for word in words if word not in CUSTOM_STOP_WORDS]
        return ' '.join(filtered_words)

    return ' '.join(text.split())

## Feature Builder

In [ ]:
def fit_vectorizer_experiment(train_frame, ngram_range, max_features, norm_type):
    vectorizer = CountVectorizer(
        lowercase=True,
        ngram_range=ngram_range,
        max_features=max_features,
    )

    if norm_type == 'basic':
        premise_text = train_frame['premise'].fillna('').map(normalize_basic)
        hypothesis_text = train_frame['hypothesis'].fillna('').map(normalize_basic)
    else:
        premise_text = train_frame.apply(lambda row: normalize_with_negation(row, 'premise'), axis=1)
        hypothesis_text = train_frame.apply(lambda row: normalize_with_negation(row, 'hypothesis'), axis=1)

    vectorizer.fit(pd.concat([premise_text, hypothesis_text]))
    return vectorizer


def make_features_experiment(
    frame,
    vectorizer,
    language_columns,
    use_overlap,
    use_difference,
    use_language,
    norm_type,
):
    if norm_type == 'basic':
        premise_text = frame['premise'].fillna('').map(normalize_basic)
        hypothesis_text = frame['hypothesis'].fillna('').map(normalize_basic)
    else:
        premise_text = frame.apply(lambda row: normalize_with_negation(row, 'premise'), axis=1)
        hypothesis_text = frame.apply(lambda row: normalize_with_negation(row, 'hypothesis'), axis=1)

    premise_vec = vectorizer.transform(premise_text)
    hypothesis_vec = vectorizer.transform(hypothesis_text)

    blocks = [premise_vec, hypothesis_vec]

    if use_overlap:
        blocks.append(premise_vec.multiply(hypothesis_vec))
    if use_difference:
        blocks.append(abs(premise_vec - hypothesis_vec))
    if use_language:
        lang_vec = csr_matrix(
            pd.get_dummies(frame['lang_abv'])
            .reindex(columns=language_columns, fill_value=0)
            .astype('int8')
        )
        blocks.append(lang_vec)

    return hstack(blocks, format='csr')

## Experiment Grid

In [ ]:
EXPERIMENTS = [
    {
        'run_name': 'xgboost_basic_normalization',
        'norm_type': 'basic',
        'ngram_range': (1, 1),
        'max_features': 20_000,
        'use_overlap': True,
        'use_difference': True,
        'use_language': True,
        'n_estimators': 100,
        'learning_rate': 0.1,
    },
    {
        'run_name': 'xgboost_negation_preserved_normalization',
        'norm_type': 'advanced_negation',
        'ngram_range': (1, 1),
        'max_features': 20_000,
        'use_overlap': True,
        'use_difference': True,
        'use_language': True,
        'n_estimators': 100,
        'learning_rate': 0.1,
    },
]

language_columns = sorted(train_df['lang_abv'].dropna().unique().tolist())
results = []

## Run Experiments

In [ ]:
for params in EXPERIMENTS:
    run_name = params['run_name']
    print(f'Running {run_name} (normalization: {params["norm_type"]})...')

    vectorizer = fit_vectorizer_experiment(
        train_df,
        ngram_range=params['ngram_range'],
        max_features=params['max_features'],
        norm_type=params['norm_type'],
    )

    x_train = make_features_experiment(
        train_df,
        vectorizer,
        language_columns,
        params['use_overlap'],
        params['use_difference'],
        params['use_language'],
        params['norm_type'],
    )
    x_val = make_features_experiment(
        val_df,
        vectorizer,
        language_columns,
        params['use_overlap'],
        params['use_difference'],
        params['use_language'],
        params['norm_type'],
    )
    x_test = make_features_experiment(
        test_df,
        vectorizer,
        language_columns,
        params['use_overlap'],
        params['use_difference'],
        params['use_language'],
        params['norm_type'],
    )

    model = XGBClassifier(
        random_state=42,
        n_estimators=params['n_estimators'],
        learning_rate=params['learning_rate'],
        eval_metric='mlogloss',
    )
    model.fit(x_train, train_df['label'].astype(int))

    val_preds = model.predict(x_val)
    metrics = compute_all_metrics(val_df, val_preds)
    test_preds = model.predict(x_test)

    submission = build_submission(sample_submission, test_preds)
    submission_path = Path('../submissions') / f'{run_name}.csv'
    save_submission(submission, submission_path)

    model_path = Path('../submissions') / f'{run_name}_model.joblib'
    vectorizer_path = Path('../submissions') / f'{run_name}_vectorizer.joblib'
    language_columns_path = Path('../submissions') / f'{run_name}_language_columns.json'

    joblib.dump(model, model_path)
    joblib.dump(vectorizer, vectorizer_path)
    language_columns_path.write_text(json.dumps(language_columns, indent=2), encoding='utf-8')

    serving_type = 'sklearn_engineered_text_pair_features'
    metadata = {
        'run_name': run_name,
        'data_source': data_source,
        'serving_type': serving_type,
        'label_mapping': {0: 'entailment', 1: 'neutral', 2: 'contradiction'},
        'preprocessing': params['norm_type'],
        'feature_order': ['premise_vec', 'hypothesis_vec']
        + (['overlap_vec'] if params['use_overlap'] else [])
        + (['difference_vec'] if params['use_difference'] else [])
        + (['language_one_hot'] if params['use_language'] else []),
        'language_columns': language_columns,
        'params': {**params, 'ngram_range': list(params['ngram_range'])},
    }

    mlflow_params = {
        **params,
        'ngram_range': str(params['ngram_range']),
        'model': 'XGBClassifier',
        'random_state': 42,
        'data_source': data_source,
        'serving_type': serving_type,
    }

    with start_notebook_run(
        run_name,
        tags={
            'model_family': 'xgboost',
            'features': f'bow_norm_{params["norm_type"]}',
            'serving_type': serving_type,
        },
    ):
        mlflow.log_params(mlflow_params)
        log_metrics(metrics)
        log_common_context('../configs/data_split.yaml', '../data/processed/split_metadata.json')
        mlflow.log_artifact(str(submission_path), artifact_path='submissions')
        log_model_artifacts(
            artifacts={
                'model.joblib': model_path,
                'vectorizer.joblib': vectorizer_path,
                'language_columns.json': language_columns_path,
            },
            metadata=metadata,
            artifact_path='model',
        )

    results.append({
        'run_name': run_name,
        'normalization': params['norm_type'],
        'accuracy': metrics['accuracy'],
        'f1_macro': metrics['f1_macro'],
        'precision_macro': metrics['precision_macro'],
        'recall_macro': metrics['recall_macro'],
        'submission_path': str(submission_path),
        'serving_type': serving_type,
    })

pd.DataFrame(results).sort_values('f1_macro', ascending=False)